# 02 - Data Cleaning

All issues identified in the validation notebook are addressed here.

In [1]:
# Setup
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
from settings import PATHS, apply_pandas_settings

apply_pandas_settings()

In [2]:
# Load data
customers = pd.read_csv(PATHS['raw_customers'])
loans = pd.read_csv(PATHS['raw_loans'])
transactions = pd.read_csv(PATHS['raw_transactions'])

In [3]:
customers.head()

,customer_id,account_open_date,acquisition_channel,kyc_level,location
0,CUS00001,2022-01-05,Ads,Tier 2,Lagos
1,CUS00002,2022-01-10,Loan Campaign,Tier 2,Lagos
2,CUS00003,2022-01-22,Loan Campaign,Tier 2,Lagos
3,CUS00004,2022-05-20,Ads,Tier 3,Lagos
4,CUS00005,2022-02-27,Loan Campaign,Tier 1,Plateau


In [4]:
loans.head()

,loan_id,customer_id,loan_taken,loan_amount,loan_defaulted
0,LON00001,CUS00001,No,NaN,NaN
1,LON00002,CUS00002,Yes,40000.00,No
2,LON00003,CUS00003,Yes,10000.00,No
3,LON00004,CUS00004,No,NaN,NaN
4,LON00005,CUS00005,Yes,20000.00,Yes


In [5]:
transactions.head()

,transaction_id,customer_id,transaction_type,transaction_date,remark,amount
0,TXN0021277,CUS01170,Transfer,2022-05-14 07:18:26,Transfer to Blessing Okonkwo | Access Bank | 2...,111300.00
1,TXN0030504,CUS01668,Bank Charge,2022-08-27 13:19:08,ATM usage fee,35.00
2,TXN0052726,CUS02941,VAT,2022-07-12 11:13:38,VAT on transaction fee,7.50
3,TXN0044119,CUS02475,Airtime,2022-06-29 07:39:20,9mobile airtime recharge to 08124622954,750.00
4,TXN0017473,CUS00944,VAT,2022-07-01 09:26:20,VAT on transaction fee,7.50


In [6]:
# Convert date columns to datetime
customers['account_open_date'] = pd.to_datetime(customers['account_open_date'])
transactions['transaction_date'] = pd.to_datetime(transactions['transaction_date'])

In [7]:
display(customers.dtypes)
display(transactions.dtypes)

customer_id                       str
account_open_date      datetime64[us]
acquisition_channel               str
kyc_level                         str
location                          str
dtype: object

transaction_id                 str
customer_id                    str
transaction_type               str
transaction_date    datetime64[us]
remark                         str
amount                     float64
dtype: object

In [8]:
# Remove duplicate transactions
transactions.duplicated().sum()

np.int64(31)

In [9]:
transactions = transactions.drop_duplicates(keep='first')

In [10]:
# Verify no duplicates remain
transactions.duplicated().sum()

np.int64(0)

In [11]:
# Standardise acquisition channel
acquisition_channel_dict = {
    'loan campaign': 'Loan Campaign',
    'referral': 'Referral',
    'ref': 'Referral',
    'ads': 'Ads',
    'ad': 'Ads',
    'agnt ntwk': 'Agent Network',
    'agent network': 'Agent Network',
    'direct': 'Direct',
    'dir': 'Direct',
}

customers['acquisition_channel'] = (
    customers['acquisition_channel'].str.strip().str.lower().map(acquisition_channel_dict)
)

In [12]:
# Verify standardised acquisition channels
customers['acquisition_channel'].value_counts()

acquisition_channel
Loan Campaign    2014
Ads              1468
Referral          741
Agent Network     439
Direct            338
Name: count, dtype: int64

In [13]:
# No NaN values from mismatch error
customers['acquisition_channel'].isnull().sum()

np.int64(0)

In [14]:
# Resolve loan_taken (No) and loan_defaulted (Yes) contradiction
loans[(loans['loan_taken'] == 'No') & (loans['loan_defaulted'] == 'Yes')]

,loan_id,customer_id,loan_taken,loan_amount,loan_defaulted
59,LON00060,CUS00060,No,NaN,Yes
529,LON00530,CUS00530,No,NaN,Yes
558,LON00559,CUS00559,No,NaN,Yes
761,LON00762,CUS00762,No,NaN,Yes
834,LON00835,CUS00835,No,NaN,Yes
1270,LON01271,CUS01271,No,NaN,Yes
1334,LON01335,CUS01335,No,NaN,Yes
1569,LON01570,CUS01570,No,NaN,Yes
1688,LON01689,CUS01689,No,NaN,Yes
1694,LON01695,CUS01695,No,NaN,Yes


In [15]:
# Compare null values in loan_amount and loan_defaulted
loan_amount_nulls = loans['loan_amount'].isnull().sum()
loan_defaulted_nulls = loans['loan_defaulted'].isnull().sum()

print(f'loan amount nulls before fix: {loan_amount_nulls}')
print(f'loan defaulted nulls before fix: {loan_defaulted_nulls}')

loan amount nulls before fix: 2205
loan defaulted nulls before fix: 2175


In [16]:
# Set loan_defaulted to NaN where loan_taken = No
loans.loc[loans['loan_taken'] == 'No', 'loan_defaulted'] = np.nan

In [17]:
# Verify null counts are now consistent
loan_amount_nulls = loans['loan_amount'].isnull().sum()
loan_defaulted_nulls = loans['loan_defaulted'].isnull().sum()

print(f'loan amount nulls after fix: {loan_amount_nulls}')
print(f'loan defaulted nulls after fix: {loan_defaulted_nulls}')

loan amount nulls after fix: 2205
loan defaulted nulls after fix: 2205


In [18]:
# Handle outlier transaction amounts
transactions['transaction_type'].value_counts()

transaction_type
VAT                   21935
Savings               13024
Airtime               12999
Bill Pay              12993
Transfer               7676
Transfer Fee           7672
Stamp Duty             7212
Reversal               1501
Card Payment - POS     1280
Bank Charge            1273
Card Payment - ATM     1273
Name: count, dtype: int64

In [19]:
# Verify that no negative transaction amount exist
negative_transactions_amounts = transactions[transactions['amount'] < 0]
print(f'transactions with negative amounts: {len(negative_transactions_amounts)}')

transactions with negative amounts: 0


In [20]:
system_transaction_types = [
    'VAT', 'Transfer Fee', 'Stamp Duty', 'Reversal', 'Bank Charge'
]

user_transactions = transactions[~transactions['transaction_type'].isin(system_transaction_types)]

outlier_ids = []
for txn_type in user_transactions['transaction_type'].dropna().unique():
    user_transactions_amounts = user_transactions[user_transactions['transaction_type'] == txn_type]['amount']
    Q1 = user_transactions_amounts.quantile(0.25)
    Q3 = user_transactions_amounts.quantile(0.75)

    IQR = Q3 - Q1
    upper_bound = Q3 + 3 * IQR

    outliers = user_transactions[(user_transactions['transaction_type'] == txn_type) \
                                 & (user_transactions['amount'] > upper_bound)]
    
    fintech_outliers = outliers[outliers['amount'] > upper_bound * 10]
    outlier_ids.extend(fintech_outliers['transaction_id'].to_list())


print(f'Outlier transaction ids flagged: {len(outlier_ids)}')
print(outlier_ids)




Outlier transaction ids flagged: 7
['TXN9000013', 'TXN9000014', 'TXN9000012', 'TXN9000002', 'TXN9000003', 'TXN9000004', 'TXN9000005']


In [21]:
# Check upper bound per transaction type
for txn_type in user_transactions['transaction_type'].dropna().unique():
    user_transactions_amounts = user_transactions[user_transactions['transaction_type'] == txn_type]['amount']
    Q1 = user_transactions_amounts.quantile(0.25)
    Q3 = user_transactions_amounts.quantile(0.75)

    IQR = Q3 - Q1
    upper_bound = Q3 + 3 * IQR
    print(f'{txn_type:<24} upper bound:{upper_bound:>12.2f}  *10:{upper_bound*10:>14.2f}')

Transfer                 upper bound:   336650.00  *10:    3366500.00
Airtime                  upper bound:     4850.00  *10:      48500.00
Bill Pay                 upper bound:   111450.00  *10:    1114500.00
Card Payment - POS       upper bound:   439400.00  *10:    4394000.00
Savings                  upper bound:   223950.00  *10:    2239500.00
Card Payment - ATM       upper bound:   344000.00  *10:    3440000.00


In [22]:
# Use a conservative IQR approach to preserve business context
system_transaction_types = [
    'VAT', 'Transfer Fee', 'Stamp Duty', 'Reversal', 'Bank Charge'
]

user_transactions = transactions[~transactions['transaction_type'].isin(system_transaction_types)]

outlier_ids = []
outlier_rows = []
for txn_type in user_transactions['transaction_type'].dropna().unique():
    user_transactions_amounts = user_transactions[user_transactions['transaction_type'] == txn_type]['amount']
    Q1 = user_transactions_amounts.quantile(0.25)
    Q3 = user_transactions_amounts.quantile(0.75)

    IQR = Q3 - Q1
    upper_bound = Q3 + 3 * IQR

    outliers = user_transactions[(user_transactions['transaction_type'] == txn_type) \
                                 & (user_transactions['amount'] > upper_bound)]
    
    fintech_outliers = outliers[outliers['amount'] > upper_bound * 2.5]
    outlier_rows.append(fintech_outliers)
    outlier_ids.extend(fintech_outliers['transaction_id'].to_list())


print(f'Outlier transaction ids flagged: {len(outlier_ids)}')
print(outlier_ids)




Outlier transaction ids flagged: 15
['TXN9000006', 'TXN9000007', 'TXN9000008', 'TXN9000009', 'TXN9000013', 'TXN9000014', 'TXN9000010', 'TXN9000011', 'TXN9000012', 'TXN9000000', 'TXN9000001', 'TXN9000002', 'TXN9000003', 'TXN9000004', 'TXN9000005']


In [23]:
# Verify outlier transaction amounts
fintech_outlier_rows = pd.concat(outlier_rows)
print(f'fintech outlier row count: {len(fintech_outlier_rows)}')
display(
    fintech_outlier_rows[['transaction_id', 'customer_id', 'transaction_type', 'amount']]
    .sort_values('transaction_type')
)

fintech outlier row count: 15


,transaction_id,customer_id,transaction_type,amount
88867,TXN9000013,CUS04543,Airtime,75000.00
88868,TXN9000014,CUS01999,Airtime,120000.00
88864,TXN9000010,CUS00990,Bill Pay,500000.00
88865,TXN9000011,CUS04260,Bill Pay,850000.00
88866,TXN9000012,CUS00641,Bill Pay,1200000.00
88854,TXN9000000,CUS00609,Savings,800000.00
88855,TXN9000001,CUS01400,Savings,1200000.00
88856,TXN9000002,CUS00970,Savings,2500000.00
88857,TXN9000003,CUS02059,Savings,3800000.00
88858,TXN9000004,CUS01497,Savings,4200000.00


In [24]:
fintech_outlier_rows

,transaction_id,customer_id,transaction_type,transaction_date,remark,amount
88860,TXN9000006,CUS03653,Transfer,2022-06-25 20:53:47,Transfer to test | Access Bank | 0000000000,900000.00
88861,TXN9000007,CUS01522,Transfer,2022-11-19 08:46:17,Transfer to test | Access Bank | 0000000000,1500000.00
88862,TXN9000008,CUS04364,Transfer,2022-10-27 21:02:55,Transfer to test | Access Bank | 0000000000,2200000.00
88863,TXN9000009,CUS00651,Transfer,2022-10-09 12:46:21,Transfer to test | Access Bank | 0000000000,3000000.00
88867,TXN9000013,CUS04543,Airtime,2022-03-11 06:15:04,MTN airtime recharge to 08000000000,75000.00
88868,TXN9000014,CUS01999,Airtime,2022-06-16 22:44:58,MTN airtime recharge to 08000000000,120000.00
88864,TXN9000010,CUS00990,Bill Pay,2022-09-10 07:23:34,DSTV Subscription payment,500000.00
88865,TXN9000011,CUS04260,Bill Pay,2022-11-29 06:19:20,DSTV Subscription payment,850000.00
88866,TXN9000012,CUS00641,Bill Pay,2022-06-05 14:17:09,DSTV Subscription payment,1200000.00
88854,TXN9000000,CUS00609,Savings,2022-05-25 21:21:05,Savings deposit,800000.00


In [25]:
# Remove outlier transaction amounts
before = len(transactions)

outlier_transaction_ids = fintech_outlier_rows['transaction_id'].unique()
transactions = transactions[~transactions['transaction_id'].isin(outlier_transaction_ids)]

after = len(transactions)

print(f'removed {before - after} outlier transactions')

removed 15 outlier transactions


In [26]:
# Verify no outlier transactions remain
transactions['transaction_id'].isin(outlier_transaction_ids).sum()

np.int64(0)

In [27]:
# Remove system-generated transactions
transactions['transaction_type'].value_counts()

transaction_type
VAT                   21935
Savings               13018
Airtime               12997
Bill Pay              12990
Transfer               7672
Transfer Fee           7672
Stamp Duty             7212
Reversal               1501
Card Payment - POS     1280
Bank Charge            1273
Card Payment - ATM     1273
Name: count, dtype: int64

In [28]:
transactions = transactions[~transactions['transaction_type'].isin(system_transaction_types)]

In [29]:
# Verify that system-generated transactions no longer exist
transactions['transaction_type'].value_counts()

transaction_type
Savings               13018
Airtime               12997
Bill Pay              12990
Transfer               7672
Card Payment - POS     1280
Card Payment - ATM     1273
Name: count, dtype: int64

In [30]:
# Save clean datasets
customers.to_csv(PATHS['clean_customers'], index = False)
loans.to_csv(PATHS['clean_loans'], index = False)
transactions.to_csv(PATHS['clean_transactions'], index = False)

In [31]:
# Final data quality check
print('CUSTOMERS')
print(f'  rows          : {len(customers):,}')
print(f'  columns       : {list(customers.columns)}')
print(f'  nulls         : {customers.isnull().sum().sum()}')
print(f'  account_open_date dtype : {customers['account_open_date'].dtype}')
print(f'  acquisition_channel unique values : {sorted(customers['acquisition_channel'].unique())}')


print('\nLOANS')
print(f'  rows           : {len(loans):,}')
print(f'  columns        : {list(loans.columns)}')
loans_mismatch = loans[(loans['loan_taken'] == 'No') & (loans['loan_defaulted'] == 'Yes')]
print(f'  loan_defaulted = Yes where loan_taken = No: {len(loans_mismatch)}')
print(f'  loan_amount nulls: {loans['loan_amount'].isnull().sum():,}')
print(f'  loan_defaulted nulls: {loans['loan_defaulted'].isnull().sum():,}')


print('\nTRANSACTIONS')
print(f'  rows           : {len(transactions):,}')
print(f'  columns        : {list(transactions.columns)}')
print(f'  nulls          : {transactions.isnull().sum().sum()}')
print(f'  transaction_date dtype : {transactions['transaction_date'].dtype}')
print(f'  duplicate rows : {transactions.duplicated().sum()}')
print(f'\ntransaction types after removing system-generated transactions:')
print(transactions['transaction_type'].value_counts().to_string())


CUSTOMERS
  rows          : 5,000
  columns       : ['customer_id', 'account_open_date', 'acquisition_channel', 'kyc_level', 'location']
  nulls         : 0
  account_open_date dtype : datetime64[us]
  acquisition_channel unique values : ['Ads', 'Agent Network', 'Direct', 'Loan Campaign', 'Referral']

LOANS
  rows           : 5,000
  columns        : ['loan_id', 'customer_id', 'loan_taken', 'loan_amount', 'loan_defaulted']
  loan_defaulted = Yes where loan_taken = No: 0
  loan_amount nulls: 2,205
  loan_defaulted nulls: 2,205

TRANSACTIONS
  rows           : 49,230
  columns        : ['transaction_id', 'customer_id', 'transaction_type', 'transaction_date', 'remark', 'amount']
  nulls          : 0
  transaction_date dtype : datetime64[us]
  duplicate rows : 0

transaction types after removing system-generated transactions:
transaction_type
Savings               13018
Airtime               12997
Bill Pay              12990
Transfer               7672
Card Payment - POS     1280
Card Paym